# BITS PILANI WILP
## Course: Software Engineering for Machine Learning (AIMLCZG546)
### Assignment I: Requirements Engineering and System Architecture for ML

**Group Number:** 84  
**Date:** June 8, 2026

#### **Group Members & Contributions:**

| Sl. No | BITS ID | Name | Qualitative Contribution | Quantitative Contribution (%) |
|---|---|---|---|---|
| 1 | **2025AA05710** | Singh Pritesh Sanjay Poonam | Quality Attribute Testing, Performance Benchmarking, and System Integration | 25% |
| 2 | **2025AA05368** | Gangera Tushar Kantibhai Dayaben | Requirements Formulation, Problem Statement definition, and GR4ML Modeling | 25% |
| 3 | **2025AB05154** | Gangam Shuba Nandini | Feature Engineering, ML Pipeline Design, and Model Training | 25% |
| 4 | **2025AA05574** | Shaifali Garg | System Architecture Diagram, FastAPI Serving, and Event Simulation | 25% |

---


## Objective 1: Requirements Formulation

### 1. Domain & Problem Statement
* **Domain:** Consumer Credit Risk Assessment & Automated Loan Underwriting.
* **Problem Statement:** 
  Lending institutions process thousands of loan applications daily. Manual review of these applications is slow, costly, and prone to subjective biases. We design a real-time, Machine Learning-based decision support system to automate the underwriting process by predicting the **default risk** of each loan applicant. Given an applicant's financial profile (income, credit score, employment, assets, liabilities, and loan details), the system must predict if the application should be approved or denied, while assigning a structured risk tier (LOW, MEDIUM, or HIGH) and maintaining a latency of < 150ms.

* **Dataset:** [Financial Risk for Loan Approval](https://www.kaggle.com/datasets/lorenzozoppelletto/financial-risk-for-loan-approval) — 20,000 real-world loan application records with 36 features covering demographics, credit history, employment, and financial metrics.

### 2. GR4ML Requirement Specifications & Goals

**GR4ML** (Goal-Oriented Requirements Engineering for Machine Learning) organizes our specifications into three views:

#### **A. Business View**
Aligns the high-level business goals with the ML requirements.
* **Actors:** Credit Officer (reviews flagged exceptions), Loan Applicant (submits financial details).
* **Strategic Goals:** Minimize default losses (Risk Control), automate underwriting (Operational Efficiency).
* **Decision Goals:** Approve, Deny, or Flag application for manual audit.
* **Question Goals:** Is the credit risk acceptable? Does the applicant meet debt-to-income thresholds?
* **Indicators:** Default Rate < 2.5%, Auto-approval rate > 80%, response latency < 150ms.
* **Insights:** Default probability score, risk tier (LOW/MEDIUM/HIGH), computed net worth.

#### **B. Analytics Design View**
Translates business questions into algorithm selection and softgoals.
* **Analytics Goal:** Prediction (Binary classification of default risk).
* **Algorithm:** RandomForestClassifier (n_estimators=150, max_depth=10) — robust ensemble method that handles non-linear relationships and provides built-in feature importance rankings. Achieved 90.7% accuracy on the test set.
* **Softgoals:** Prediction Accuracy (90.7%), Inference Performance (< 150ms latency), Explainability (Feature Importance ranking), and Input Reliability (Pydantic type validation).

#### **C. Data Preparation View**
Specifies the transformation workflows required to convert raw data into model features.
* **Data Source:** Single Kaggle CSV file (Loan.csv) containing 20,000 records with 36 features covering demographics, credit history, employment, and financial metrics.
* **Prep Tasks:**
  1. Dropping non-predictive columns (ApplicationDate, RiskScore, InterestRate, BaseInterestRate, MonthlyLoanPayment, TotalDebtToIncomeRatio) — 36 → 30 features
  2. Removing redundant features via correlation analysis (MonthlyIncome ↔ AnnualIncome r=0.99, Age ↔ Experience r=0.98, TotalAssets ↔ NetWorth r=0.98) — 30 → 27 features
  3. Removing low-importance features (MaritalStatus, HomeOwnershipStatus, NumberOfDependents, NumberOfCreditInquiries, NumberOfOpenCreditLines, UtilityBillsPaymentHistory) — 27 → 21 features
  4. Categorical encoding (dictionary mapping for EmploymentStatus, EducationLevel, LoanPurpose)
  5. Feature engineering (LoanToIncomeRatio, SavingsToLoanRatio) — 21 → 22 features + target
* **Operators:** `Dictionary Mapping`, `Correlation Matrix`, `Feature Importance Analysis`, and `Pydantic` validation.

### 3. GR4ML Graphical Notations

The following GR4ML diagrams model the three system views using the notation conventions from the GR4ML framework. Each view uses standardized node shapes (ovals for goals, dashed-border ellipses for softgoals, rectangles for tasks/insights, and stick-figure actors) and labeled relationship arrows.

---

#### **I. Business View Diagram (The "Why?")**

The Business View captures the high-level stakeholder intentions and organizational objectives that drive the ML system.

![Business View](diagrams/gr4ml_business_view.png)

**Notation & Elements:**
- **Actors:** Credit Officer and Loan Applicant, connected to strategic goals via *desires* relationships.
- **Strategic Goals:** "Minimize Loan Defaults" (risk control) and "Automate Underwriting" (operational efficiency), which *support* the central Decision Goal.
- **Decision Goal [D]:** "Approve / Deny / Flag" -- the primary ML-driven decision the system must make.
- **Question Goal [Q]:** "Is credit risk acceptable?" -- the analytical question that must be *answered* by model insights.
- **Indicators:** Default Rate < 2.5%, Auto-approval > 80%, Latency < 150ms -- the Decision Goal is *evaluated* against these measurable KPIs (shown with traffic-light status markers).
- **Insights:** Default Probability Score, Risk Tier (LOW/MED/HIGH), and Computed Net Worth -- *generated* by the ML model and *answering* the Question Goal. Each insight *contributes (+)* to the corresponding softgoals (Accuracy, Performance, Explainability).

---

#### **II. Analytics Design View Diagram (The "What?")**

The Analytics Design View translates the business questions into algorithm selection and defines quality trade-offs via softgoals.

![Analytics Design View](diagrams/gr4ml_analytics_design_view.png)

**Notation & Elements:**
- **PredictionGoal:** "Predict Default Risk (Binary Classification)" -- shown as a split-pill oval, indicating the analytics type is *prediction*.
- **Algorithm:** Random Forest Classifier (n_estimators=150, max_depth=10), connected to the PredictionGoal via an *achieves* relationship. This is the sole implemented algorithm, chosen for its high accuracy (90.7%) and built-in feature importance capabilities.
- **Softgoals (Quality Attributes):** Accuracy (90.7%), Performance (< 150ms), Explainability (Feature Importance), and Reliability (Pydantic) -- shown as dashed-border ellipses (cloud shapes).
- **Influence Links:** The algorithm's contribution to softgoals is labeled with the standard GR4ML influence notation: `++` (Make -- strong positive) for Accuracy and Explainability, `+` (Help -- positive) for Performance and Reliability. Random Forest *makes (++)* Accuracy through ensemble voting and *makes (++)* Explainability through built-in feature importance scores (top predictor: LoanToIncomeRatio at 0.3654).
- **Insight:** "Feature Importance Ranking" is *generated* from the Explainability softgoal, feeding back into the Business View's Question Goal.

---

#### **III. Data Preparation View Diagram (The "How?")**

The Data Preparation View specifies the end-to-end data transformation pipeline, linking each preparation task to its corresponding data quality goal.

![Data Prep View](diagrams/gr4ml_data_prep_view.png)

**Notation & Elements:**
- **Data Source (top):** Single Kaggle CSV file (Loan.csv, 20,000 records, 36 features) forms the *start* of the data flow.
- **Sequential Prep Tasks:** The pipeline flows top-to-bottom through five stages:
  1. **Drop Non-Predictive Columns** (ApplicationDate, RiskScore, InterestRate, etc.) — reduces from 36 to 30 features.
  2. **Remove Redundant Features** via correlation analysis (threshold r > 0.95: MonthlyIncome, Experience, TotalAssets) — reduces from 30 to 27 features.
  3. **Remove Low-Importance Features** (importance < 0.005: MaritalStatus, HomeOwnership, NumberOfDependents, etc.) — reduces from 27 to 21 features.
  4. **Categorical Encoding** (dictionary mapping for EmploymentStatus, EducationLevel, LoanPurpose).
  5. **Feature Engineering** (LoanToIncomeRatio, SavingsToLoanRatio) — expands from 21 to 22 features + target.
  
  Each task is shown with a double-bottom-line rectangle (GR4ML Data Preparation Task notation).
- **Data Quality Softgoals (right column):** Each prep task is linked via dashed *operationalizes* or influence (`++`/`+`) arrows to its corresponding quality goal: Relevant Features (no noise), No Multicollinearity (independent features), Parsimonious Model (no redundancy), Homogeneous Encoding, and Rich Feature Representation.
- **Output:** The pipeline terminates at the processed dataset (`loan_data_processed.csv`, pd.DataFrame [20000, 22+1]).

### 4. Top Three Quality Requirements

We identify the following three non-functional quality requirements as critical for the loan underwriting system:

1. **Robustness (Data Validation & Boundary Protection)**
   * **Justification:** ML models fail silently when fed out-of-distribution or corrupted data. By enforcing strict boundaries (e.g. Credit score must be 300-850, age must be >= 18) via Pydantic schemas, we protect the downstream estimator from garbage-in-garbage-out behavior.
   * **Measurable Metric:** 100% of invalid data payloads rejected with explicit validation errors before model execution.

2. **Reliability (Deterministic Tiers & Fault Tolerance)**
   * **Justification:** Loan decisions are subject to strict financial regulations. The system must output consistent classification probability bounds and structured risk categories (LOW/MEDIUM/HIGH) without crash risk under high concurrent loads.
   * **Measurable Metric:** Service availability uptime >= 99.99%, zero unhandled 500 server errors on model predictions.

3. **Performance (Low Latency Inference)**
   * **Justification:** Real-time loan decisioning is embedded directly in client-facing online application portals. High inference latencies directly lead to application drop-offs and poor user engagement.
   * **Measurable Metric:** Average model prediction pipeline response time < 150ms (SLA bound).


## Objective 2: System Architecture

### 5. System Architecture Diagram

The system architecture combines **Sculley's "Hidden Technical Debt" layout** (separating configuration, logging, serving infrastructure, and monitoring) with the **Pipe-and-Filter execution pipeline**:

![System Architecture](diagrams/system_architecture.png)

---

### 6 & 7. Selection & Implementation of Two Architectural Patterns

#### **Pattern 1: Pipe-and-Filter Pattern**
* **Application:** Implemented in `app/pipeline.py`. The prediction sequence is structured as a series of isolated filters that consume a specific input type, perform transformations, and pipe their output to the next stage.
  * **Filter 1 (validate_input):** Enforces business rules (Robustness).
  * **Filter 2 (extract_features):** Computes financial ratios and encodes features (Maintainability).
  * **Filter 3 (run_model):** Performs RandomForest prediction scoring (Performance).
  * **Filter 4 (format_response):** Maps probabilities to risk tiers (Reliability).

#### **Pattern 2: Microservices Serving & Event-Driven Logging Pattern**
* **Application:** Implemented in `app/main.py` and `app/logger.py`. The ML model is served via a FastAPI microservice exposing REST APIs. The service uses structured JSON logging with runtime attributes (latency, decision, features) emitted to stdout for downstream log aggregators (Elastic/Splunk), and includes a `/health` endpoint for orchestrator liveness/readiness checks.

## ML Pipeline Code Execution & Verification

In [ ]:
# Load and preprocess the Kaggle dataset
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder

# Load raw data
raw_path = "data/Loan.csv"
df_raw = pd.read_csv(raw_path)
print("Raw Dataset Shape:", df_raw.shape)
print("\nFirst 5 records:")
display(df_raw.head())

# Preprocessing: Drop non-predictive columns
drop_cols = ['ApplicationDate', 'RiskScore', 'BaseInterestRate',
             'InterestRate', 'MonthlyLoanPayment', 'TotalDebtToIncomeRatio']
df = df_raw.drop(columns=drop_cols)

# Remove redundant features (high multicollinearity)
df = df.drop(columns=['MonthlyIncome', 'Experience', 'TotalAssets'])

# Remove low-importance features
df = df.drop(columns=['MaritalStatus', 'HomeOwnershipStatus', 'NumberOfDependents',
                      'NumberOfCreditInquiries', 'NumberOfOpenCreditLines',
                      'UtilityBillsPaymentHistory'])

# Encode categorical variables
for col in ['EmploymentStatus', 'EducationLevel', 'LoanPurpose']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    print(f"Encoded '{col}': {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Feature Engineering
df['LoanToIncomeRatio'] = df['LoanAmount'] / (df['AnnualIncome'] + 1)
df['SavingsToLoanRatio'] = df['SavingsAccountBalance'] / (df['LoanAmount'] + 1)

print(f"\nProcessed Dataset Shape: {df.shape}")
print(f"\nTarget Distribution (LoanApproved):")
print(df['LoanApproved'].value_counts())
print(f"\nFinal Features ({df.shape[1] - 1}):")
print([c for c in df.columns if c != 'LoanApproved'])

In [ ]:
# Train Random Forest model on processed data
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

target_col = 'LoanApproved'
feature_cols = [c for c in df.columns if c != target_col]
X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training Random Forest (n_estimators=150, max_depth=10)...")
print(f"Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples\n")

model = RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, preds), 4))
print("\nClassification Report:")
print(classification_report(y_test, preds))

# Top 10 Feature Importances
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Top 10 Feature Importances:")
for feat, imp in importances.head(10).items():
    print(f"  {feat}: {imp:.4f}")

# Save model for serving
os.makedirs("app", exist_ok=True)
joblib.dump(model, "app/model.pkl")
print("\nModel saved to app/model.pkl")

In [ ]:
# Demonstrating Pipe-and-Filter execution programmatically
from app.schemas import LoanApplicationInput
from app.pipeline import execute_pipeline
import joblib

# Load trained model
clf = joblib.load("app/model.pkl")

# Low-risk applicant (high income, good credit, low loan)
low_risk_app = LoanApplicationInput(
    age=45, annual_income=95000, credit_score=720, employment_status=0,
    education_level=4, loan_amount=25000, loan_duration=36,
    monthly_debt_payments=400, credit_card_utilization_rate=0.25,
    debt_to_income_ratio=0.15, bankruptcy_history=0, loan_purpose=3,
    previous_loan_defaults=0, payment_history=28, length_of_credit_history=15,
    savings_account_balance=35000, checking_account_balance=8000,
    total_liabilities=30000, job_tenure=10, net_worth=220000
)

# High-risk applicant (low income, bad credit, high loan)
high_risk_app = LoanApplicationInput(
    age=23, annual_income=28000, credit_score=480, employment_status=2,
    education_level=3, loan_amount=40000, loan_duration=60,
    monthly_debt_payments=900, credit_card_utilization_rate=0.88,
    debt_to_income_ratio=0.65, bankruptcy_history=1, loan_purpose=4,
    previous_loan_defaults=1, payment_history=12, length_of_credit_history=2,
    savings_account_balance=200, checking_account_balance=500,
    total_liabilities=45000, job_tenure=1, net_worth=-10000
)

print("--- PIPE-AND-FILTER: LOW-RISK APPLICANT ---")
result_low = execute_pipeline(low_risk_app, clf)
print(result_low.model_dump_json(indent=2))

print("\n--- PIPE-AND-FILTER: HIGH-RISK APPLICANT ---")
result_high = execute_pipeline(high_risk_app, clf)
print(result_high.model_dump_json(indent=2))

print("\n--- BUSINESS RULE REJECTION (Loan > 500% of income) ---")
try:
    rejected_app = LoanApplicationInput(
        age=28, annual_income=20000, credit_score=600, employment_status=1,
        education_level=1, loan_amount=120000, loan_duration=60,
        monthly_debt_payments=500, credit_card_utilization_rate=0.50,
        debt_to_income_ratio=0.40, bankruptcy_history=0, loan_purpose=3,
        previous_loan_defaults=0, payment_history=20, length_of_credit_history=5,
        savings_account_balance=2000, checking_account_balance=1000,
        total_liabilities=15000, job_tenure=3, net_worth=10000
    )
    execute_pipeline(rejected_app, clf)
except ValueError as e:
    print(f"Caught business rule violation: {e}")

In [ ]:
# Verify the FastAPI Serving infrastructure using FastAPI TestClient
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)

print("--- TESTING GET /health ---")
response_health = client.get("/health")
print("Status Code:", response_health.status_code)
print("Response:", response_health.json())

print("\n--- TESTING POST /predict (Valid Payload) ---")
payload_valid = {
    "age": 45, "annual_income": 95000, "credit_score": 720,
    "employment_status": 0, "education_level": 4, "loan_amount": 25000,
    "loan_duration": 36, "monthly_debt_payments": 400,
    "credit_card_utilization_rate": 0.25, "debt_to_income_ratio": 0.15,
    "bankruptcy_history": 0, "loan_purpose": 3, "previous_loan_defaults": 0,
    "payment_history": 28, "length_of_credit_history": 15,
    "savings_account_balance": 35000, "checking_account_balance": 8000,
    "total_liabilities": 30000, "job_tenure": 10, "net_worth": 220000
}
response_pred = client.post("/predict", json=payload_valid)
print("Status Code:", response_pred.status_code)
print("Response JSON:")
print(response_pred.json())

print("\n--- TESTING POST /predict (Invalid Schema - credit_score > 850) ---")
payload_invalid = payload_valid.copy()
payload_invalid["credit_score"] = 950
response_invalid = client.post("/predict", json=payload_invalid)
print("Status Code (Expected 422):", response_invalid.status_code)
print("Response Detail:")
print(response_invalid.json()["detail"][0]["msg"])

### 8. Application Screenshots & API Demonstration

The deployed FastAPI microservice exposes a Swagger UI (`/docs`) for interactive API testing. Below are screenshots demonstrating a live prediction request and response.

#### **Request: POST /predict**

The Swagger UI allows sending a JSON payload with all applicant features to the `/predict` endpoint:

![Swagger Request](diagrams/swagger_request.png)

The request body includes 20 features covering demographics (`age`, `employment_status`, `education_level`, `job_tenure`), financial profile (`annual_income`, `credit_score`, `monthly_debt_payments`, `savings_account_balance`, `checking_account_balance`, `total_liabilities`, `net_worth`), loan details (`loan_amount`, `loan_duration`, `loan_purpose`), credit behavior (`credit_card_utilization_rate`, `debt_to_income_ratio`, `bankruptcy_history`, `previous_loan_defaults`, `payment_history`, `length_of_credit_history`). All fields are validated against Pydantic boundary constraints (e.g., credit_score must be 300-850, age >= 18) before reaching the ML model.

#### **Response: 200 OK**

The server returns a structured JSON response containing the prediction decision, probability, risk tier, computed financial ratios, and inference latency:

![Swagger Response](diagrams/swagger_response.png)

**Key observations from the response:**
- `is_approved: true` -- the applicant is approved based on the model's probability threshold.
- `probability: 0.8435` -- the RandomForest model predicts an 84.35% likelihood of loan repayment.
- `risk_tier: "LOW"` -- mapped deterministically from the probability score.
- `net_worth: 220000` -- passed as an input feature representing total assets minus total liabilities.
- `debt_to_income: 0.15` -- the debt-to-income ratio as provided in the applicant's profile.
- `latency_ms: 52.04` -- well within the 150ms SLA bound, confirming the Performance quality requirement.

This demonstrates the end-to-end Pipe-and-Filter pattern: the request passes through validation (Filter 1), feature extraction (Filter 2), model scoring (Filter 3), and response formatting (Filter 4) -- each stage operating as an isolated filter in the pipeline.

---

### Summary & Conclusion

Through this assignment, we successfully engineered a machine learning system for automated loan underwriting:
1. **Requirements Formulation:** Developed goals and indicators using the **GR4ML** conceptual framework, mapping the Business, Analytics, and Data Preparation views. The Business View captures *why* the system exists (strategic goals, decision goals, question goals, indicators, and insights). The Analytics Design View specifies *what* analytics to apply (PredictionGoal, RandomForest algorithm selection, softgoal trade-offs with influence links). The Data Preparation View defines *how* raw data is transformed (5 sequential pipeline tasks — drop non-predictive, remove redundant, remove low-importance, encode categoricals, engineer features — linked to data quality softgoals via operationalization and contribution relationships).
2. **System Architecture Design:** Integrated Sculley's ML systems framework (separating data validation, configuration, and JSON logging) with a **Pipe-and-Filter pattern** for deterministic transaction execution and a **Microservices Serving pattern** via FastAPI.
3. **Execution & Validation:** Built a FastAPI server to serve predictions and verified system stability across key quality attributes: **Robustness** (boundary protection via Pydantic across 20 input features), **Reliability** (consistent schemas and deterministic risk tiers), and **Performance** (latency well under the 150ms SLA).
4. **Live API Demonstration:** Swagger UI screenshots confirm end-to-end system functionality with structured JSON input (20 validated features) and output (approval decision, probability, risk tier, financial ratios), validating all three quality requirements in production-like conditions.